# Training a model for the CM4AI DREAM Challenge

This tutorial walks through training a model on challenge data, checking how well it does, and turning it into a submission. The model under test is **BLAST-ProtiCelli**, the challenge's example model; each step shows where a user-defined model would take its place.

BLAST-ProtiCelli takes a protein's amino acid sequence and three reference images of a cell, and predicts an image of where that protein sits in the cell. How it does that is not the subject here; the repository's README has the details.

![Tutorial flow: steps 1 to 5 train a model in this notebook and repeat until it is ready; step 6 submits it](../docs/tutorial_flow.svg)

**Phase A, in this notebook: train a model**

1. **Set up:** settings (1a), install the model (1b), check the GPU (1c), choose the data (1d), and get the starting weights (1e).
2. **Format the data:** download training cells and convert them to the form the model reads.
3. **Train:** train the model further on the new cells, then package it as a weights file.
4. **Predict** held-out cells with the starting and the trained weights.
5. **Evaluate** both with `cm4ai-evaluate`, then decide: add data, change the model, or submit.

**Phase B, on the command line: submit.** The weights file from step 3 goes into the model's workflow, `blast_proticelli.wdl`, unchanged; a separate tutorial covers a test run on one or two cells.

The training here is deliberately small. The goal is to show where a user-defined model fits, not to improve the example model; see the notes in step 3 before reading anything into the results.

**Runs on** an AnVIL (Terra) Jupyter environment with a GPU, and also on a local machine or cluster.


## 1. Set up

### 1a. Settings

The notebook's control panel. Start with `RUN_MODE = "quick"`, which runs every step on a small sample to check that everything works; switch to `"full"` for a real training run. The training and prediction values belong to the example model; a user-defined model would bring its own.

Everything the notebook creates goes under `WORK_DIR`, by default the repository's `training/work` folder, so deleting that folder resets the notebook.


In [ ]:
import os
from pathlib import Path

RUN_MODE = "quick"          # "quick": a small run that checks every step; "full": a real training run
# Sets the antibodies, the split, training and predictions. A number repeats a run exactly (quick mode uses 1,
# so reruns reuse its downloads); None takes a new seed from the clock, saved in WORK_DIR so reruns keep it.
SEED = {"quick": 1, "full": None}[RUN_MODE]
WORK_DIR = None             # everything this notebook writes; None: the repository's training/work folder

# This repository's code. Used only when no copy is found next to the notebook or in the home folder
# (for example, the notebook uploaded on its own to a Terra workspace): the repository is then cloned
# into the home folder.
REPO_URL = "https://github.com/cm4ai-anvil-dream-challenge/blast-proticelli.git"
REPO_BRANCH = "main"

# Optional: an existing BLAST-ProtiCelli weights zip (for example blast_proticelli_weights_v2.zip).
# Its model folders are reused instead of downloading them again (about 6.5 GB).
# Read from the environment variable BLAST_PROTICELLI_WEIGHTS_ZIP if set, or type a path here.
# A gs:// path (for example a zip in an AnVIL workspace bucket) is copied once into WORK_DIR/downloads.
# The UniProt reference data is always rebuilt, so the new weights file gets the current gene map.
EXISTING_WEIGHTS_ZIP = os.environ.get("BLAST_PROTICELLI_WEIGHTS_ZIP")

# Training (step 3), values for the example model. Training all of it (about 460M parameters) needs
# roughly 8 GB of GPU memory before images; lower BATCH_SIZE if training runs out of memory.
EPOCHS = {"quick": 1, "full": 20}[RUN_MODE]
BATCH_SIZE = 4
LEARNING_RATE = 1e-5        # peak; the model's own default is 1e-4, gentler here for a small dataset
RETRAIN = False             # False: reuse this run's fine-tuned weights (same data set, mode and seed), if present

# Prediction (step 4): diffusion steps per prediction, and how many held-out cells to compare (None: all)
PREDICTION_STEPS = {"quick": 20, "full": 50}[RUN_MODE]
COMPARE_CELLS = {"quick": 12, "full": None}[RUN_MODE]

print(f"Settings: RUN_MODE={RUN_MODE}, epochs={EPOCHS}, batch size={BATCH_SIZE}")


### 1b. Install the model

Installs the example model's code and everything it needs, once per environment. In a submission, the model's Docker image does this; here, without Docker, the notebook does it.

The code is this repository, BLAST-ProtiCelli. `pip install` also brings in its Python dependencies, including the ProtiCelli package it builds on (pinned in `pyproject.toml`). On AnVIL this takes several minutes, mostly for PyTorch; afterwards, restart the kernel and run from the top.

The notebook normally sits in the `training/` folder of a clone of the repository:

```bash
git clone https://github.com/cm4ai-anvil-dream-challenge/blast-proticelli.git
```

Opened anywhere else (for example uploaded to a Terra workspace's Analyses tab), it uses a `blast-proticelli` folder in the home directory, cloning `REPO_BRANCH` of `REPO_URL` there if it is missing.


In [ ]:
import subprocess, sys

def is_repo(d):
    return (d / "pyproject.toml").is_file() and (d / "training" / "build_weights_zip.sh").is_file()

REPO_DIR = next((d for d in [Path.cwd().parent, Path.cwd() / "blast-proticelli", Path.home() / "blast-proticelli"]
                 if is_repo(d)), None)
if REPO_DIR is None:
    REPO_DIR = Path.home() / "blast-proticelli"
    print(f"No clone of the repository found; cloning {REPO_URL} ({REPO_BRANCH}) into {REPO_DIR}")
    proc = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                          capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if proc.returncode != 0:
        raise RuntimeError(f"Could not clone {REPO_URL} ({REPO_BRANCH}). A private repository needs credentials; "
                           f"check REPO_URL and REPO_BRANCH in 1a.\n{proc.stderr.strip()}")
    assert is_repo(REPO_DIR), f"{REPO_DIR} does not look like the blast-proticelli repository"
print("Repository:", REPO_DIR)

WORK_DIR = Path(WORK_DIR).expanduser().resolve() if WORK_DIR else REPO_DIR / "training" / "work"
DOWNLOAD_DIR = WORK_DIR / "downloads"    # starting weights, reference data
print("Work folder:", WORK_DIR)

import time
seed_file = WORK_DIR / f"seed_{RUN_MODE}.txt"
if SEED is None:
    SEED = int(seed_file.read_text()) if seed_file.is_file() else int(time.time()) % 2**31
WORK_DIR.mkdir(parents=True, exist_ok=True)
seed_file.write_text(str(SEED))
print(f"Seed: {SEED} (saved in {seed_file}; set SEED in 1a to repeat a run)")

from importlib import metadata

try:
    print("Model already installed: blast-proticelli", metadata.version("blast-proticelli"))
except metadata.PackageNotFoundError:
    # pip's output is printed here as it arrives (several minutes, mostly PyTorch); lines about packages
    # that are already installed are left out
    proc = subprocess.Popen([sys.executable, "-m", "pip", "install", "--progress-bar", "off", str(REPO_DIR)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if not line.startswith("Requirement already satisfied"):
            print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError("pip install failed; see the output above")
    print("Installed. Restart the kernel and run from the top.")


The example model also uses BLAST+, NCBI's sequence search program, to identify proteins from their sequence. BLAST+ is a compiled program rather than a Python package, so pip cannot install it; `training/install_blast.sh` does, on Linux (including AnVIL), pinned to the same version as the Docker image. On a Mac, install it first with `conda install -c bioconda blast`.


In [ ]:
import shutil

if not (shutil.which("blastp") and shutil.which("makeblastdb")):
    proc = subprocess.run(["bash", str(REPO_DIR / "training" / "install_blast.sh"), str(WORK_DIR / "tools")],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip())
    os.environ["PATH"] = proc.stdout.strip().splitlines()[-1] + os.pathsep + os.environ["PATH"]
print(subprocess.run(["blastp", "-version"], capture_output=True, text=True).stdout.splitlines()[0])


### 1c. Check the GPU

Training needs a GPU to finish in reasonable time. On AnVIL, the GPU is chosen when the cloud environment is created; if this cell reports CPU, recreate the environment with a GPU attached.


In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("CUDA GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB)")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple GPU (MPS)")
else:
    DEVICE = "cpu"
    print("No GPU found: training will be very slow. Fine for checking the steps run, not for real training.")
print("PyTorch", torch.__version__)

### 1d. Choose the data

Two choices: **which data set** to train on, and **where to get it**. Every location holds the same files in the same layout, so the rest of the notebook works the same whichever is chosen.

**Data set**

| `DATASET` | What it is |
|---|---|
| `"KOLF-train_v1.0"` (default) | CM4AI images of KOLF2.1J human induced pluripotent stem cells, new to the example model |
| `"HPA-train_v1.0"` | Human Protein Atlas images in 39 cell lines; overlaps the example model's original training data |

**Data source**

| `DATA_SOURCE` | Where | Notes |
|---|---|---|
| `"anvil"` (default) | The challenge's AnVIL bucket | Requester pays, billed to the AnVIL workspace's project (free within AnVIL's US central region). Available until the end of the challenge |
| `"huggingface"` | [LundbergLab/CM4AI-DREAM](https://huggingface.co/datasets/LundbergLab/CM4AI-DREAM) | Public, no login needed. **Use this on a local machine or cluster**, and after the challenge |
| `"cfde"` | CFDE Cloud Workbench | Not yet tested |
| `"local"` | An existing folder | A copy downloaded earlier, or user-defined data in the same layout: the manifest, plus `tars/` or extracted antibody folders |

Only the files the notebook needs are downloaded: the manifest, the license, and one archive per antibody it uses. Both data sets are licensed CC BY 4.0, with different attributions; step 2 shows the license.


In [ ]:
DATASET = "KOLF-train_v1.0"     # "KOLF-train_v1.0" or "HPA-train_v1.0"
DATA_SOURCE = "anvil"           # "anvil", "huggingface", "cfde" or "local". Not on AnVIL? Use "huggingface".

# Where each source keeps the data. Only the chosen source's entry is used.
ANVIL_BUCKET = "gs://fc-7533ae9b-e06e-4297-9bd5-a4dbc0895420"
HF_REPO = "LundbergLab/CM4AI-DREAM"
CFDE_LOCATION = None            # not yet tested
LOCAL_DATA_DIR = None           # for "local": the data set's folder, for example "~/data/KOLF-train_v1.0"

# The AnVIL bucket is requester pays: downloads are billed to this Google Cloud project. AnVIL sets
# GOOGLE_PROJECT to the workspace's project; elsewhere, set it before starting Jupyter.
BILLING_PROJECT = os.environ.get("GOOGLE_PROJECT")

# How much of the data set to use
N_TRAIN_ANTIBODIES = {"quick": 10, "full": 500}[RUN_MODE]
MAX_TRAIN_CELLS = {"quick": 200, "full": None}[RUN_MODE]     # None: every cell of the training antibodies
N_TEST_ANTIBODIES = {"quick": 3, "full": 50}[RUN_MODE]
SPLIT_BY = "antibody"       # "antibody", "fov" or "cell": which cells may not be split between training and held-out sets

print(f"Data: {DATASET} from {DATA_SOURCE}; {N_TRAIN_ANTIBODIES} training and {N_TEST_ANTIBODIES} held-out antibodies")


`data_file()` below fetches one file of the data set from the chosen source, once; step 2 uses it for everything it downloads. Everything a run produces (training inputs, trained weights, predictions, scores) goes in its own run folder, named by data set, run mode and seed, so changing any of them starts fresh rather than reusing another run's results. There is nothing to change here.


In [ ]:
MANIFEST_NAME = {"KOLF-train_v1.0": "kolf_train_manifest.csv.gz",
                 "HPA-train_v1.0": "hpa_train_manifest.csv.gz"}[DATASET]
DATA_DIR = WORK_DIR / "data" / DATASET          # downloads and extracted antibody folders
LOCAL_DATA = Path(LOCAL_DATA_DIR).expanduser() if LOCAL_DATA_DIR else None
DATA_DIR.mkdir(parents=True, exist_ok=True)

def data_file(rel):
    """Local path of one file of the data set (rel is relative to the data set's folder), downloading it if needed."""
    for root in [LOCAL_DATA, DATA_DIR]:
        if root and (root / rel).is_file():
            return root / rel
    dest = DATA_DIR / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if DATA_SOURCE == "anvil":
        if not BILLING_PROJECT:
            raise RuntimeError("The AnVIL bucket is requester pays: set GOOGLE_PROJECT before starting Jupyter "
                               "(AnVIL sets it automatically), or use DATA_SOURCE = \"huggingface\".")
        uri = f"{ANVIL_BUCKET}/{DATASET}/{rel}"
        if shutil.which("gcloud"):
            cmd = ["gcloud", "storage", "cp", f"--billing-project={BILLING_PROJECT}", uri, str(dest)]
        else:
            cmd = ["gsutil", "-q", "-u", BILLING_PROJECT, "cp", uri, str(dest)]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode != 0:
            raise RuntimeError(f"Download failed: {uri}\n{proc.stderr.strip()}")
    elif DATA_SOURCE == "huggingface":
        from huggingface_hub import hf_hub_download    # installed with the model's dependencies
        hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=f"{DATASET}/{rel}",
                        local_dir=WORK_DIR / "data")    # saves to DATA_DIR / rel
    elif DATA_SOURCE == "cfde":
        raise NotImplementedError("The CFDE Cloud Workbench source is not yet tested. Download the data set "
                                  "there first and use DATA_SOURCE = \"local\" with LOCAL_DATA_DIR.")
    elif DATA_SOURCE == "local":
        raise FileNotFoundError(f"{rel} is not in LOCAL_DATA_DIR ({LOCAL_DATA})")
    else:
        raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE!r}")
    return dest

RUN_DIR = WORK_DIR / "runs" / f"{DATASET}_{RUN_MODE}_seed{SEED}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data folder: {DATA_DIR}")
print(f"Run folder: {RUN_DIR}")

# A gentle reminder: each run keeps its own trained weights and training inputs, which add up
def folder_gb(d):
    return sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1e9

older = [d for d in (WORK_DIR / "runs").iterdir() if d.is_dir() and d != RUN_DIR]
if older:
    older_gb = sum(folder_gb(d) for d in older)
    print(f"\n{len(older)} other run folder(s) in {WORK_DIR / 'runs'}, {older_gb:.1f} GB in all:")
    for d in sorted(older):
        print(f"  {d.name}")
    if len(older) > 1 or older_gb > 3:
        print("Delete any no longer needed; downloads and data in WORK_DIR are kept separately and are not affected.")


### 1e. Get the starting weights

Training starts from the example model's published weights, and the model's weights file also carries the reference data it uses to identify proteins. Two downloads, both kept under `WORK_DIR/downloads`:

* **Starting weights:** ProtiCelli's published weights from the Lundberg lab (`checkpoint/` and `vae/`, about 6.5 GB), unless `EXISTING_WEIGHTS_ZIP` is set or a previous run already fetched them.
* **Reference data:** the reviewed human proteins from UniProt as a BLAST database, and a gene map from UniProt accessions and Ensembl protein IDs to HGNC symbols. These are built by `build_human_blast_db.sh` and `build_human_gene_map.sh` in this folder, which can also be run on their own. They always rebuild from the current UniProt release, which is recorded in each zip's `PROVENANCE.txt`.

A download that fails stops the cell with the reason; rerunning the cell picks up where it left off.


In [ ]:
import shutil
import zipfile

def local_weights_zip(src):
    # A local path is used as is; a gs:// path is copied once into DOWNLOAD_DIR.
    if not str(src).startswith("gs://"):
        return Path(src).expanduser()
    local = DOWNLOAD_DIR / str(src).rstrip("/").split("/")[-1]
    if local.is_file():
        print("Reusing", local)
        return local
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    partial = local.with_name(local.name + ".part")
    billing = [f"--billing-project={BILLING_PROJECT}"] if BILLING_PROJECT else []
    print(f"Copying {src} to {local} ...", flush=True)
    proc = subprocess.run(["gcloud", "storage", "cp", *billing, str(src), str(partial)], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Could not copy {src}\n{proc.stderr.strip()}")
    partial.rename(local)
    return local

if EXISTING_WEIGHTS_ZIP:
    EXISTING_WEIGHTS_ZIP = str(local_weights_zip(EXISTING_WEIGHTS_ZIP))   # step 4 also reads it, as the starting weights

sys.path.insert(0, str(REPO_DIR))      # use this repository's code, not an older installed copy
from proticelli.utils.download import checkpoint_ready, vae_ready, download_checkpoints

PROTICELLI_DIR = DOWNLOAD_DIR / "proticelli"     # will hold checkpoint/ and vae/
TRAINING_SCRIPTS = REPO_DIR / "training"

def proticelli_weights_ready(d):
    # fit() starts from checkpoint/unet_ema; inference loads checkpoint/unet. Require both, plus the VAE.
    ema = d / "checkpoint" / "unet_ema"
    return checkpoint_ready(d) and vae_ready(d) and (ema / "config.json").is_file()

if proticelli_weights_ready(PROTICELLI_DIR):
    print("Reusing ProtiCelli weights in", PROTICELLI_DIR)
elif EXISTING_WEIGHTS_ZIP:
    print("Extracting ProtiCelli weights from", EXISTING_WEIGHTS_ZIP)
    with zipfile.ZipFile(Path(EXISTING_WEIGHTS_ZIP).expanduser()) as zf:
        members = [n for n in zf.namelist() if n.startswith(("checkpoint/unet/", "checkpoint/unet_ema/", "vae/"))]
        zf.extractall(PROTICELLI_DIR, members=members)
else:
    download_checkpoints(PROTICELLI_DIR)

assert proticelli_weights_ready(PROTICELLI_DIR), f"ProtiCelli weights in {PROTICELLI_DIR} are incomplete"
print("ProtiCelli weights ready")

In [ ]:
REF_DIR = DOWNLOAD_DIR / "reference"
BLAST_DB_DIR = REF_DIR / "human_blast_db"
GENE_MAP_DIR = REF_DIR / "human_gene_map"

def build_reference(script, out_dir, product):
    if (out_dir / product).is_file():
        print(f"Reusing {out_dir / product}")
        return
    print(f"Running {script} (downloads from UniProt; a few minutes)")
    proc = subprocess.run(["bash", str(TRAINING_SCRIPTS / script), str(out_dir)], capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        raise RuntimeError(f"{script} failed:\n{proc.stderr[-2000:]}")

build_reference("build_human_blast_db.sh", BLAST_DB_DIR, "human_blast_db.zip")
build_reference("build_human_gene_map.sh", GENE_MAP_DIR, "human_gene_map.zip")
for d in (BLAST_DB_DIR, GENE_MAP_DIR):
    print((d / "PROVENANCE.txt").read_text())

## 2. Format the data

This step turns the data set chosen in 1d into what the model reads: it downloads the cells it needs, matches their proteins to the model, splits them into training and held-out sets, and converts the training cells to the model's input format.

Both data sets share one layout, described on the [dataset card](https://huggingface.co/datasets/LundbergLab/CM4AI-DREAM): a manifest with one row per cell (antibody, gene, Ensembl IDs, cell line, field of view, and the path of each file), one archive per antibody under `tars/`, and a license. Every cell has the five 512 x 512 PNGs the challenge uses (protein, nucleus, microtubules, ER, and a cell mask), and every antibody has a FASTA file with its target's amino acid sequence. **Take identifiers and paths from the manifest's columns, never from file names** (cell line names contain hyphens).

**What the example model already knows.** ProtiCelli was trained on Human Protein Atlas images, which overlap HPA-train, so training on HPA-train shows *how* a team trains rather than improving the model. KOLF-train's images are new to it. In either data set, only proteins already in the model's vocabulary can be trained (2b).


In [ ]:
import json
import tarfile
import numpy as np
import pandas as pd

# Terms of use for the images
try:
    print(data_file("LICENSE.txt").read_text())
except RuntimeError as err:
    print(f"LICENSE.txt not shown: {err}")

In [ ]:
# The manifest: one row per cell
manifest_path = data_file(MANIFEST_NAME)
manifest = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
print(f"{manifest_path}: {len(manifest):,} cells, {manifest['antibody_id'].nunique():,} antibodies, "
      f"{manifest['gene_name'].nunique():,} genes\n")

per_antibody = manifest.groupby("antibody_id").size()
print(f"Cells per antibody: median {per_antibody.median():.0f}, range {per_antibody.min()} to {per_antibody.max()}")
print(f"Cells per field of view: median {manifest.groupby('fov_key').size().median():.0f}\n")
manifest.head(3)

### 2a. Downloading cell images

The images are stored as one archive (`.tar`) per antibody, holding the images of every cell stained with that antibody plus the FASTA file of its target protein. `fetch_antibodies()` downloads and extracts the archives for a list of antibodies into `WORK_DIR/data/<DATASET>`, skipping any already there (or in `LOCAL_DATA_DIR`), and checks that every file the manifest lists for them arrived. Step 2c uses it for the training and held-out antibodies; here it fetches one archive of typical size, to look at one of its cells.


In [ ]:
def antibody_root(antibody_id):
    # Folder holding the extracted <antibody_id>/ folder, or None if it is not present yet.
    for root in [LOCAL_DATA, DATA_DIR]:
        if root and (root / antibody_id).is_dir():
            return root
    return None

def fetch_antibodies(antibody_ids):
    rows = manifest[manifest["antibody_id"].isin(antibody_ids)]
    tars = rows.drop_duplicates("antibody_id").set_index("antibody_id")["tar_file"]
    missing = sorted(set(antibody_ids) - set(tars.index))
    if missing:
        raise KeyError(f"Not in the manifest: {missing}")
    fetched = 0
    for i, (antibody_id, tar_name) in enumerate(tars.items(), 1):
        if antibody_root(antibody_id):
            continue
        tar_path = data_file(f"tars/{antibody_id[:5]}/{tar_name}")
        with tarfile.open(tar_path) as t:
            try:
                t.extractall(DATA_DIR, filter="data")   # refuses unsafe paths (Python 3.12, and 3.11.4 or later)
            except TypeError:
                t.extractall(DATA_DIR)
        if tar_path.is_relative_to(DATA_DIR):
            tar_path.unlink()    # a downloaded copy; never delete a tar in LOCAL_DATA_DIR
        fetched += 1
        print(f"  [{i}/{len(tars)}] {antibody_id}")
    # Every file the manifest lists for these antibodies must now exist
    path_columns = ["protein_path", "nucleus_path", "microtubules_path", "er_path", "mask_path", "sequence_path"]
    absent = [p for _, r in rows.iterrows() for p in (antibody_root(r["antibody_id"]) / r[c] for c in path_columns) if not p.is_file()]
    if absent:
        raise FileNotFoundError(f"{len(absent)} files listed in the manifest are missing, for example {absent[0]}")
    print(f"{len(tars)} antibodies ready ({fetched} downloaded), {len(rows)} cells")
    return rows

def cell_files(row):
    # Absolute paths of one cell's files, from its manifest row.
    root = antibody_root(row["antibody_id"])
    return {c[: -len("_path")]: root / row[c] for c in ["protein_path", "nucleus_path", "microtubules_path", "er_path", "mask_path", "sequence_path"]}

print("fetch_antibodies() and cell_files() ready")


In [ ]:
import matplotlib.pyplot as plt
from skimage.io import imread

def scaled(img):
    # 0 to 1, stretched to the image's 99.9th percentile so dim stains are visible
    img = img.astype(np.float32)
    top = np.percentile(img, 99.9)
    return np.clip(img / top, 0, 1) if top > 0 else img

def reference_composite(microtubules, nucleus, er):
    # The usual immunofluorescence colors: nucleus blue, microtubules red, ER yellow
    mt, nu, e = scaled(microtubules), scaled(nucleus), scaled(er)
    return np.clip(np.stack([mt + e, e, nu], axis=-1), 0, 1)

def protein_green(protein, top=None):
    # Protein in green on black; pass top to put several images on one scale
    p = protein.astype(np.float32) / (top or max(np.percentile(protein, 99.9), 1))
    return np.stack([np.zeros_like(p), np.clip(p, 0, 1), np.zeros_like(p)], axis=-1)

EXAMPLE_ANTIBODY = (per_antibody - per_antibody.median()).abs().idxmin()   # a typical antibody, by number of cells
example = fetch_antibodies([EXAMPLE_ANTIBODY]).iloc[0]
files = cell_files(example)

img = {ch: imread(files[ch]) for ch in ["microtubules", "nucleus", "er", "protein", "mask"]}
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
axes[0].imshow(reference_composite(img["microtubules"], img["nucleus"], img["er"]))
axes[0].set_title("reference: nucleus (blue), microtubules (red), ER (yellow)", fontsize=9)
axes[1].imshow(protein_green(img["protein"]))
axes[1].set_title(f"protein: {example['gene_name']}", fontsize=9)
axes[2].imshow(protein_green(img["protein"]))
axes[2].contour(img["mask"] > 0, levels=[0.5], colors="white", linewidths=0.8)
axes[2].set_title("protein, with the cell mask outlined", fontsize=9)
for ax in axes:
    ax.axis("off")
fig.suptitle(f"{example['image_key']}: the reference channels go in, the protein image is what a model predicts")
plt.show()
print(files["sequence"].read_text()[:300])

### 2b. Matching proteins to the model

The example model learns one embedding per protein, so it can only train on cells whose protein is in its vocabulary (`antibody_map.pkl` in the installed ProtiCelli package). This check matters: an antibody whose protein is not in the vocabulary cannot be used for training, so it is left out and listed.

One naming difference is handled here: antibodies against several related proteins. The manifests list their genes as `TPM1;TPM3`; ProtiCelli has one entry for the group, `TPM1,TPM3`. They are matched as sets of names.

Adding new proteins means enlarging the model's protein embedding table and training the new rows: one of the places a team could take this further. (At prediction, the example model still handles an unknown protein by using the closest protein it knows.)


In [ ]:
import pickle
from importlib import resources
import protein_predictor as pp

proticelli_data = resources.files("proticelli") / "data"
with open(proticelli_data / "antibody_map.pkl", "rb") as f:
    protein_keys = list(pickle.load(f))
vocabulary = pp.Vocabulary(protein_keys)
groups = {frozenset(t.strip() for t in k.split(",")): k for k in protein_keys}

def protein_key(gene_name):
    # The model's vocabulary entry for a manifest gene name, or None.
    if ";" in gene_name:
        return groups.get(frozenset(g.strip() for g in gene_name.split(";")))
    return vocabulary.resolve(gene_name)

antibodies = manifest.drop_duplicates("antibody_id")[["antibody_id", "gene_name"]].copy()
antibodies["protein_key"] = antibodies["gene_name"].map(protein_key)

usable = antibodies["protein_key"].notna()
print(f"Proteins: {usable.sum():,} of {len(antibodies):,} antibodies match ProtiCelli's vocabulary "
      f"({(antibodies.loc[usable, 'gene_name'] != antibodies.loc[usable, 'protein_key']).sum()} of them as a group of genes)")

not_in_vocabulary = antibodies[~usable]
not_in_vocabulary.to_csv(RUN_DIR / "antibodies_not_in_vocabulary.csv", index=False)
print(f"\n{len(not_in_vocabulary)} antibodies cannot be used (listed in {RUN_DIR / 'antibodies_not_in_vocabulary.csv'}), for example:")
not_in_vocabulary[["antibody_id", "gene_name"]].head(8)

### 2c. Choosing antibodies and splitting

`N_TRAIN_ANTIBODIES + N_TEST_ANTIBODIES` antibodies are drawn at random (seeded by `SEED`) from those the model knows. Their cells are then split into a training set and a held-out set, which steps 4 and 5 use to compare the starting and the trained weights. The first run downloads every chosen antibody's archive, which can take several minutes; later runs reuse them.

**How to split** (`SPLIT_BY`) is a real choice. Cells of one antibody, and above all cells from one field of view, look much alike: same staining, same imaging session, often neighboring cells. If near copies end up on both sides, the held-out score partly measures memory. So:

* `"antibody"` (default): every cell of an antibody is on the same side, so held-out proteins were never seen in training. This matches the challenge, which asks for proteins in new cells.
* `"fov"`: fields of view are kept together, but an antibody can be on both sides.
* `"cell"`: cells are split individually. Fastest to learn from, and the most likely to overstate performance.

In [ ]:
rng = np.random.default_rng(SEED)
pool = antibodies.loc[usable, "antibody_id"].to_numpy()
if len(pool) < N_TRAIN_ANTIBODIES + N_TEST_ANTIBODIES:
    raise ValueError(f"Only {len(pool)} usable antibodies; lower N_TRAIN_ANTIBODIES or N_TEST_ANTIBODIES in 1d")
chosen = rng.choice(pool, size=N_TRAIN_ANTIBODIES + N_TEST_ANTIBODIES, replace=False)

cells = manifest[manifest["antibody_id"].isin(chosen)].copy()
cells["protein_key"] = cells["antibody_id"].map(antibodies.set_index("antibody_id")["protein_key"])

if SPLIT_BY == "antibody":
    held_out_groups = set(chosen[:N_TEST_ANTIBODIES])
    group_column = "antibody_id"
elif SPLIT_BY in ("fov", "cell"):
    group_column = "fov_key" if SPLIT_BY == "fov" else "image_key"
    group_ids = cells[group_column].unique()
    rng.shuffle(group_ids)
    n_held_out = round(len(group_ids) * N_TEST_ANTIBODIES / (N_TRAIN_ANTIBODIES + N_TEST_ANTIBODIES))
    held_out_groups = set(group_ids[:n_held_out])
else:
    raise ValueError(f'SPLIT_BY must be "antibody", "fov" or "cell", not {SPLIT_BY!r}')
cells["split"] = np.where(cells[group_column].isin(held_out_groups), "test", "train")

print(f"{len(chosen)} antibodies, {len(cells):,} cells, split by {SPLIT_BY}:")
print(cells.groupby("split").agg(cells=("image_key", "size"), antibodies=("antibody_id", "nunique")).to_string(), "\n")

fetch_antibodies(list(chosen))
cells.to_csv(RUN_DIR / "training_cells.csv", index=False)
print(f"Written to {RUN_DIR / 'training_cells.csv'}")
cells.drop_duplicates("antibody_id")[["antibody_id", "gene_name", "protein_key", "split"]].sort_values("split")

### 2d. Formatting the training cells for the model

Each training cell's four images are combined and normalized into the single input the example model trains on, using the model's own tools: the same normalization it applies at prediction time, so training and prediction see identically scaled images. A user-defined model would replace this step with its own input format.

Held-out cells are not converted: step 4 predicts them from the original images.


In [ ]:
import tifffile
from image_scaling import read_png       # checks the challenge format: single-channel 8-bit PNG, 512 x 512
from proticelli.data import ChannelAssembler, ImageNormalizer

TRAIN_DIR = RUN_DIR / "train_inputs"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
assembler, normalizer = ChannelAssembler(has_protein=True), ImageNormalizer(bit_depth=8)
CHANNELS = ("microtubules", "nucleus", "er", "protein")

train_cells = cells[cells["split"] == "train"]
if MAX_TRAIN_CELLS and len(train_cells) > MAX_TRAIN_CELLS:
    train_cells = train_cells.sample(MAX_TRAIN_CELLS, random_state=SEED).sort_index()
    print(f"Using {MAX_TRAIN_CELLS} of the training antibodies' cells (MAX_TRAIN_CELLS)")
built = 0
for i, (_, row) in enumerate(train_cells.iterrows(), 1):
    out = TRAIN_DIR / f"{row['image_key']}.tiff"
    if out.exists():
        continue
    files = cell_files(row)
    raw = assembler.transform({ch: read_png(files[ch]) for ch in CHANNELS})
    stack = normalizer.transform(raw).astype(np.float32)
    # fit() rescales any image whose maximum is above 1 as if it were raw counts; normalized images never are
    assert stack.shape == (512, 512, 4) and stack.min() >= -1 and stack.max() <= 1, row["image_key"]
    tifffile.imwrite(out, stack, compression="zlib")
    built += 1
    if i % 200 == 0:
        print(f"  {i} of {len(train_cells)}")

TRAIN_FILES = [f"{k}.tiff" for k in train_cells["image_key"]]
TRAIN_PROTEINS = train_cells["protein_key"].tolist()
size_mb = sum((TRAIN_DIR / f).stat().st_size for f in TRAIN_FILES) / 1e6
print(f"{len(TRAIN_FILES)} training inputs in {TRAIN_DIR} ({built} built now, {size_mb:.0f} MB), "
      f"{train_cells['protein_key'].nunique()} proteins")

## 3. Train

### 3a. Train the model

The example model trains with its own `fit()`, starting from the weights fetched in 1e. Every step, a batch of real protein images is corrupted with noise of a random strength, and the model learns to recover them from the three reference images and the protein's name. The plot below updates as training runs: each point is one step's loss, and the line is a running average.

**This is where a user-defined model goes.** Everything before this cell prepares the challenge data; everything after it packages and evaluates whatever weights this cell produces.

**Reading the plot.** The per-step loss jumps around because each step draws a different noise strength; the running average is the thing to watch. A falling average means the model is fitting the training images, not that its predictions for new images got better; steps 4 and 5 compare the starting and trained weights on held-out cells for that. In `"quick"` mode, expect a short, noisy curve.

**What to expect.** HPA-train overlaps the example model's own training data, so training on it changes little. KOLF-train's images are new to the model, so differences in step 5 mean more, though a quick run is too short to expect much.

**How long.** Run time is steps x seconds per step, and steps = epochs x training cells / `BATCH_SIZE`. Measured so far: about 17 seconds per step on an Apple M1-class laptop GPU and about 5 on an AnVIL T4, at batch size 4; `"quick"` mode (200 cells, 50 steps) takes about 15 minutes on the laptop and 4 on AnVIL. The cell prints its own timing when it finishes.

**Memory.** Training all of the example model's weights needs roughly 8 GB of GPU memory. On a laptop, that comes out of the same memory as everything else: close large applications first, and if Activity Monitor (Memory tab) shows swap growing, training slows down many times over. A smaller `BATCH_SIZE` helps a little; the weights and optimizer state are most of the memory.

**Weights.** ProtiCelli ships two copies of its weights: `checkpoint/unet`, which prediction loads, and `checkpoint/unet_ema`, a smoothed running average from its original training. `fit()` starts from `unet_ema` and, as configured here (EMA averaging off), saves the trained weights to a folder named `unet_ema`; 3b packages them as `checkpoint/unet` so prediction uses them.

**Reusing a run.** With `RETRAIN = False`, weights already trained in this run folder (same data set, run mode and seed) are reused: this cell then skips training and redraws that run's loss curve, so the notebook can be rerun from the top without training again.


In [ ]:
import datetime
import math
from IPython.display import display
from proticelli import Model
import proticelli._training as proticelli_training

import time

FINETUNE_DIR = RUN_DIR / "finetune"
RUN_RECORD = FINETUNE_DIR / "training_run.json"
LOSS_FILE = FINETUNE_DIR / "training_loss.csv"
steps_per_epoch = math.ceil(len(TRAIN_FILES) / BATCH_SIZE)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(1, total_steps // 10)   # fit()'s default of 500 would outlast a short run

fig, ax = plt.subplots(figsize=(9, 3.5))
plot_handle = display(fig, display_id=True)
plt.close(fig)
step_losses = []

def redraw(title=None):
    ax.clear()
    ax.plot(range(1, len(step_losses) + 1), step_losses, ".", color="0.7", markersize=3, label="loss per step")
    if len(step_losses) >= 2:
        window = min(max(5, steps_per_epoch // 10), len(step_losses))
        smooth = np.convolve(step_losses, np.ones(window) / window, mode="valid")
        ax.plot(range(window, len(step_losses) + 1), smooth, color="C0", label=f"average over {window} steps")
    for epoch in range(1, len(step_losses) // steps_per_epoch + 1):
        ax.axvline(epoch * steps_per_epoch + 0.5, color="0.9", zorder=0)
    ax.set_xlabel("training step" + (" (gray lines: epoch boundaries)" if total_steps > steps_per_epoch else ""))
    ax.set_ylabel("loss")
    ax.set_xlim(0, max(total_steps, len(step_losses)) + 1)
    ax.legend(loc="upper right")
    if title:
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    plot_handle.update(fig)

reuse = not RETRAIN and (FINETUNE_DIR / "unet_ema" / "config.json").is_file()
if reuse:
    # Earlier run: its weights, loss curve and record stay together in FINETUNE_DIR
    old_loss = LOSS_FILE if LOSS_FILE.is_file() else WORK_DIR / "training_loss.csv"   # runs before the record existed
    record = json.loads(RUN_RECORD.read_text()) if RUN_RECORD.is_file() else {}
    step_losses = pd.read_csv(old_loss)["loss"].tolist() if old_loss.is_file() else []
    steps_per_epoch = record.get("steps_per_epoch", len(step_losses) or 1)
    total_steps = len(step_losses)
    print("Reusing fine-tuned weights from an earlier run (set RETRAIN = True to train again):")
    print(json.dumps(record, indent=2) if record else "  (run details not recorded: run made before this notebook kept them)")
    redraw("earlier run" + (f", {record['finished']}" if record else ""))
else:
    print(f"{len(TRAIN_FILES)} images, batch size {BATCH_SIZE}: {steps_per_epoch} steps per epoch, "
          f"{total_steps} steps, learning rate warmup {warmup_steps} steps")
    if FINETUNE_DIR.exists():
        shutil.rmtree(FINETUNE_DIR)          # weights, loss and record are replaced together
    model = Model(checkpoint_dir=PROTICELLI_DIR / "checkpoint", vae_dir=PROTICELLI_DIR / "vae", device=DEVICE)
    torch.manual_seed(SEED)

    # Live loss plot. fit() reports each step's loss to its progress bar; a progress bar
    # that also records the loss lets the plot follow training without changing fit().
    class LossRecordingProgressBar(proticelli_training.tqdm):
        def set_postfix(self, *args, **kwargs):
            if "loss" in kwargs:
                step_losses.append(float(kwargs["loss"]))
                if len(step_losses) % max(1, steps_per_epoch // 5) == 0 or len(step_losses) == total_steps:
                    redraw()
            return super().set_postfix(*args, **kwargs)

    original_progress_bar = proticelli_training.tqdm
    proticelli_training.tqdm = LossRecordingProgressBar
    started = time.perf_counter()
    try:
        model.fit(
            image_dir=TRAIN_DIR,
            image_files=TRAIN_FILES,
            protein_names=TRAIN_PROTEINS,
            output_dir=str(FINETUNE_DIR),
            num_epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            lr_warmup_steps=warmup_steps,
            save_model_epochs=10,
            checkpointing_steps=10**9,                    # no intermediate checkpoints (1.8 GB each)
            num_workers=4 if DEVICE == "cuda" else 0,     # worker processes are unreliable in notebooks on macOS
        )
    finally:
        proticelli_training.tqdm = original_progress_bar
    seconds = time.perf_counter() - started
    redraw()

    assert (FINETUNE_DIR / "unet_ema" / "config.json").is_file(), "fit() did not save weights"
    pd.Series(step_losses, name="loss").rename_axis("step").to_csv(LOSS_FILE)
    record = {"finished": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), "run_mode": RUN_MODE, "device": DEVICE,
              "training_cells": len(TRAIN_FILES), "proteins": len(set(TRAIN_PROTEINS)), "epochs": EPOCHS,
              "batch_size": BATCH_SIZE, "steps_per_epoch": steps_per_epoch, "steps": len(step_losses),
              "learning_rate": LEARNING_RATE, "minutes": round(seconds / 60, 1),
              "seconds_per_step": round(seconds / max(len(step_losses), 1), 1)}
    RUN_RECORD.write_text(json.dumps(record, indent=2))
    print(f"Training took {record['minutes']} minutes ({record['seconds_per_step']} s per step)")
    print("Fine-tuned weights saved in", FINETUNE_DIR / "unet_ema")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

### 3b. Package the weights file

The new weights file has the same layout as the starting one, so `blast_proticelli.wdl` and `run_inference.py` use it unchanged:

```
checkpoint/unet/                trained weights
vae/                            image encoder (not trained here, copied as is)
reference/human_blast_db.zip    from 1e
reference/human_gene_map.zip    from 1e
```

It is built by `build_weights_zip.sh`, the same script used for the starting weights file. On AnVIL, it is then copied to the workspace bucket, where a workflow can use it.

In [ ]:
import datetime

PACKAGE_DIR = RUN_DIR / "package"
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
shutil.copytree(FINETUNE_DIR / "unet_ema", PACKAGE_DIR / "checkpoint" / "unet")
shutil.copytree(PROTICELLI_DIR / "vae", PACKAGE_DIR / "vae")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%MUTC")
NEW_WEIGHTS = RUN_DIR / f"blast_proticelli_weights_finetuned_{RUN_MODE}_{stamp}.zip"
proc = subprocess.run(
    ["bash", str(TRAINING_SCRIPTS / "build_weights_zip.sh"), str(PACKAGE_DIR),
     str(BLAST_DB_DIR / "human_blast_db.zip"), str(GENE_MAP_DIR / "human_gene_map.zip"), str(NEW_WEIGHTS)],
    capture_output=True, text=True)
print(proc.stdout, proc.stderr)
proc.check_returncode()
shutil.rmtree(PACKAGE_DIR)
print(f"New weights file: {NEW_WEIGHTS} ({NEW_WEIGHTS.stat().st_size / 2**30:.1f} GiB)")

# On AnVIL, keep a copy in the workspace bucket: WORK_DIR is on the notebook's own disk
bucket = os.environ.get("WORKSPACE_BUCKET")
if bucket:
    dest = f"{bucket}/weights/{NEW_WEIGHTS.name}"
    subprocess.run(["gcloud", "storage", "cp", str(NEW_WEIGHTS), dest], check=True)
    print("Copied to", dest)
else:
    print("Not on AnVIL (no WORKSPACE_BUCKET): the weights file stays in WORK_DIR; copy it somewhere permanent.")

## 4. Predict held-out cells

The held-out cells go through the baseline's own prediction script, `run_inference.py`, exactly as a workflow would run a submission: three reference PNGs and the antibody's FASTA in, one predicted PNG out. It runs twice with the same random seed: once with the **baseline** weights (ProtiCelli as published) and once with the **fine-tuned** ones. Note that, as in the challenge, the protein is identified from the FASTA sequence (by BLAST), not taken from the manifest; `prediction_results.tsv` in each output folder records which protein was used.

`COMPARE_CELLS` sets how many held-out cells to predict, spread evenly over the held-out antibodies. Measured on an Apple M1-class laptop: about 1.6 minutes per set of weights for 12 cells at 20 diffusion steps (`PREDICTION_STEPS`), including loading the model.

In [ ]:
held_out = cells[cells["split"] == "test"]
if COMPARE_CELLS and len(held_out) > COMPARE_CELLS:
    per_antibody = -(-COMPARE_CELLS // held_out["antibody_id"].nunique())   # ceiling division
    # shuffle, then take the first few cells of each antibody (keeps every column)
    held_out = (held_out.sample(frac=1, random_state=SEED)
                .groupby("antibody_id").head(per_antibody).head(COMPARE_CELLS))

records = []
for _, row in held_out.iterrows():
    f = cell_files(row)
    records.append({"prediction_id": row["image_key"], "microtubules_image": str(f["microtubules"]),
                    "nucleus_image": str(f["nucleus"]), "er_image": str(f["er"]), "fasta_file": str(f["sequence"])})
inputs_json = RUN_DIR / "held_out_inputs.json"
inputs_json.write_text(json.dumps(records, indent=2))
print(f"{len(records)} held-out cells from {held_out['antibody_id'].nunique()} antibodies")

BASELINE_WEIGHTS = Path(EXISTING_WEIGHTS_ZIP).expanduser() if EXISTING_WEIGHTS_ZIP else WORK_DIR / "blast_proticelli_weights_baseline.zip"
if not BASELINE_WEIGHTS.exists():
    print("Building the baseline weights file from the original ProtiCelli weights (several minutes)...", end=" ", flush=True)
    started = time.perf_counter()
    subprocess.run(["bash", str(TRAINING_SCRIPTS / "build_weights_zip.sh"), str(PROTICELLI_DIR),
                    str(BLAST_DB_DIR / "human_blast_db.zip"), str(GENE_MAP_DIR / "human_gene_map.zip"),
                    str(BASELINE_WEIGHTS)], check=True, capture_output=True)
    print(f"{(time.perf_counter() - started) / 60:.1f} minutes")

def predict(weights_zip, name):
    out = RUN_DIR / "predictions" / name
    cmd = [sys.executable, str(REPO_DIR / "run_inference.py"), "--inputs", str(inputs_json),
           "--weights_zip", str(weights_zip), "--output_dir", str(out),
           "--weights_dir", str(WORK_DIR / "inference_weights" / name),
           "--reference_data_dir", str(WORK_DIR / "inference_reference" / name),
           "--num_inference_steps", str(PREDICTION_STEPS), "--seed", str(SEED)]
    print(f"Predicting with the {name} weights...", end=" ", flush=True)
    started = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=WORK_DIR)
    if proc.returncode != 0:
        raise RuntimeError(proc.stdout[-3000:] + proc.stderr[-3000:])
    print(f"{(time.perf_counter() - started) / 60:.1f} minutes")
    return out

pred_dirs = {"baseline": predict(BASELINE_WEIGHTS, "baseline"), "fine-tuned": predict(NEW_WEIGHTS, "finetuned")}
used = pd.read_csv(pred_dirs["fine-tuned"] / "prediction_results.tsv", sep="\t", dtype=str)
held_out = held_out.merge(used[["prediction_id", "status", "protein_passed_to_model"]],
                          left_on="image_key", right_on="prediction_id", how="left")
print(held_out.drop_duplicates("antibody_id")[["antibody_id", "gene_name", "protein_passed_to_model", "status"]].to_string(index=False))

## 5. Evaluate

The predictions are scored against the real protein images with **cm4ai-evaluate**, the challenge's local evaluation tool, inside each cell's mask and on the whole image. Its metrics are drafts: the evaluation team is still deciding which the challenge will use and how to combine them, so treat these numbers as a demonstration of the evaluation step, not as the challenge score. No single metric is enough: what each one measures, and which direction is better, is in the [cm4ai-evaluate metric primer](https://github.com/cm4ai-anvil-dream-challenge/cm4ai-evaluate/blob/main/docs/metrics.md).

With `"quick"` settings, and with training data that overlaps ProtiCelli's own, expect small differences between the baseline and fine-tuned weights, mostly noise.

While the cm4ai-evaluate repository is private, install it once from a terminal (where git can ask for credentials) into this kernel's Python, `sys.executable`: `<that path> -m pip install "cm4ai-evaluate[extended] @ git+https://github.com/cm4ai-anvil-dream-challenge/cm4ai-evaluate.git"`.

In [ ]:
def find_evaluate():
    # On PATH, or next to this kernel's Python (an install from a terminal into the kernel's environment)
    beside = Path(sys.executable).parent / "cm4ai-evaluate"
    return shutil.which("cm4ai-evaluate") or (str(beside) if beside.is_file() else None)

if not find_evaluate():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "cm4ai-evaluate[extended] @ git+https://github.com/cm4ai-anvil-dream-challenge/cm4ai-evaluate.git"], check=True)
evaluate_cmd = find_evaluate()
assert evaluate_cmd, "cm4ai-evaluate is not installed; see the note above this cell"

EVAL_DIR = RUN_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
scores = {}
for name, d in pred_dirs.items():
    eval_manifest = EVAL_DIR / f"{name}_manifest.csv"
    pd.DataFrame({"prediction_id": held_out["image_key"],
                  "prediction": [str(d / f"{k}.png") for k in held_out["image_key"]],
                  "ground_truth": [str(cell_files(r)["protein"]) for _, r in held_out.iterrows()],
                  "mask": [str(cell_files(r)["mask"]) for _, r in held_out.iterrows()]}).to_csv(eval_manifest, index=False)
    proc = subprocess.run([evaluate_cmd, str(eval_manifest), "-o", str(EVAL_DIR / name)], capture_output=True, text=True)
    if proc.returncode == 2:
        raise RuntimeError(proc.stdout[-2000:] + proc.stderr[-2000:])
    scores[name] = pd.read_csv(EVAL_DIR / f"{name}.csv").set_index("prediction_id")
print("Scores in", EVAL_DIR)

SHOW_METRICS = ["masked_pearson_r", "masked_cosine_similarity", "masked_ms_ssim", "masked_mean_absolute_error"]
SHOW_METRICS = [m for m in SHOW_METRICS if m in scores["baseline"].columns]
summary = pd.DataFrame({name: s[SHOW_METRICS].mean() for name, s in scores.items()})
summary["change"] = summary["fine-tuned"] - summary["baseline"]
print("\nMean over held-out cells (mean absolute error: lower is better; the others: higher is better)")
summary.round(3)

In [ ]:
metric = "masked_pearson_r"
paired = pd.DataFrame({name: s[metric] for name, s in scores.items()}).join(
    held_out.set_index("image_key")[["gene_name"]])

fig, ax = plt.subplots(figsize=(5, 5))
for gene, g in paired.groupby("gene_name"):
    ax.scatter(g["baseline"], g["fine-tuned"], label=gene, s=30)
lims = [min(paired[["baseline", "fine-tuned"]].min().min(), 0), 1]
ax.plot(lims, lims, color="0.6", linewidth=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel(f"baseline model: {metric}")
ax.set_ylabel(f"fine-tuned model: {metric}")
ax.set_title("Each held-out cell; above the line means fine-tuning helped", fontsize=10)
ax.legend(fontsize=8, title="protein")
plt.show()

In [ ]:
show = held_out.groupby("antibody_id", group_keys=False).head(1)["image_key"].tolist()[:4]
fig, axes = plt.subplots(len(show), 4, figsize=(13, 3.4 * len(show)), squeeze=False)
for row_axes, key in zip(axes, show):
    r = held_out[held_out["image_key"] == key].iloc[0]
    f = cell_files(r)
    truth = imread(f["protein"])
    preds = {name: imread(pred_dirs[name] / f"{key}.png") for name in pred_dirs}
    top = max(np.percentile(truth, 99.9), 1)      # one scale for the real image and both predictions
    panels = [(f"reference channels\n{key}", reference_composite(imread(f["microtubules"]), imread(f["nucleus"]), imread(f["er"]))),
              (f"real (ground truth)\n{r['gene_name']}", protein_green(truth, top))]
    panels += [(f"{name} model prediction\nr = {scores[name].loc[key, metric]:.2f}", protein_green(p, top)) for name, p in preds.items()]
    for ax, (title, image) in zip(row_axes, panels):
        ax.imshow(image)
        ax.contour(imread(f["mask"]) > 0, levels=[0.5], colors="white", linewidths=0.6)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
fig.suptitle(f"Held-out cells: the real protein and both models' predictions, on one brightness scale (r: {metric})")
plt.tight_layout()
plt.show()

## Next steps

* **Use the weights file in a submission.** Pass the copy in the workspace bucket (`weights/` folder) as `weights_file` to `blast_proticelli.wdl`; nothing else in the workflow changes.
* **Where a team could go further:** train for longer or on more data; tune normalization or augmentation in step 2; add proteins to ProtiCelli's vocabulary (2b); replace the BLAST step with a model that reads the amino acid sequence directly; or swap ProtiCelli for another model entirely, keeping the same inputs and outputs.
* **Housekeeping.** Each run folder in `WORK_DIR/runs` holds that run's training inputs, trained weights and predictions, often a few GB; delete the ones no longer needed. `WORK_DIR/downloads` (starting weights, reference data) and `WORK_DIR/data` (downloaded cell images) are the slow downloads, worth keeping; deleting all of `WORK_DIR` resets the notebook.